In [5]:
import pandas as pd 
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle

In [6]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [8]:
label_encorder_gender=LabelEncoder()
data['Gender']=label_encorder_gender.fit_transform(data['Gender'])

In [11]:
onehot_encorder_geo=OneHotEncoder()
geo_encoded=onehot_encorder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encorder_geo.get_feature_names_out(['Geography']))
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [12]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [14]:
X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [17]:
with open('reg_label_encorder_gender.pkl','wb') as file:
    pickle.dump(label_encorder_gender,file)

with open('reg_onehot_encorder_geo.pkl','wb') as file:
    pickle.dump(onehot_encorder_geo,file)

with open('reg_scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

## ANN regression

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [21]:
model=Sequential(
    [
        Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
        Dense(32,activation='relu'),
        Dense(1) # no activation means getting same output from last layer 
    ]
)

model.compile(optimizer="adam",loss='mean_absolute_error',metrics=['mae'])

In [22]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [24]:
import os
log_dir = os.path.join(
    "regression_logs",
    "fit",
    datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
)
tensorboard_callback=TensorBoard(log_dir,histogram_freq=1)

In [25]:
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [26]:
history=model.fit(X_train,y_train,validation_data=[X_test,y_test],epochs=100,callbacks=[early_stopping_callback,tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 100398.3438 - mae: 100398.3438 - val_loss: 98595.8203 - val_mae: 98595.8203
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 99923.1797 - mae: 99923.1797 - val_loss: 97619.0078 - val_mae: 97619.0078
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 963us/step - loss: 98154.6250 - mae: 98154.6250 - val_loss: 94964.4609 - val_mae: 94964.4609
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 930us/step - loss: 94511.4219 - mae: 94511.4219 - val_loss: 90330.4609 - val_mae: 90330.4609
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 956us/step - loss: 88949.1406 - mae: 88949.1406 - val_loss: 83947.3828 - val_mae: 83947.3828
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 929us/step - loss: 81828.6484 - mae: 81828.6484 - val_loss: 76468.8438 - val_mae: 76468.8438
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 929us/step - loss: 74042.2266 - mae: 74042.2266 - val_loss: 68994.0234 - val_mae: 68994.0234
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s

In [ ]:
%load_ext tensorboard

In [35]:
%tensorboard --logdir regression_logs/fit

Reusing TensorBoard on port 6007 (pid 67614), started 0:00:05 ago. (Use '!kill 67614' to kill it.)

In [40]:
test_loss,test_metric_mae=model.evaluate(X_test,y_test)
test_metric_mae

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 627us/step - loss: 50179.6484 - mae: 50179.6484


50179.6484375

In [41]:
model.save("regression_model.h5")